In [24]:
import pandas as pd
import os
import re

In [25]:
# Basically, running the steps below will complete preparation.

# Resolve the PROJECT root (not the notebook folder)
def find_project_root(start_dir):
    current = os.path.abspath(start_dir)

    while True:
        has_main = os.path.isfile(os.path.join(current, 'main.py'))
        has_data = os.path.isdir(os.path.join(current, 'data'))

        if has_main and has_data:
            return current

        parent = os.path.dirname(current)
        if parent == current:
            return None
        current = parent


PROJECT_ROOT = find_project_root(os.getcwd())

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing both 'main.py' and 'data'."
    )

BASE_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw_data')
CLEAN_DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'clean_data')

if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"Missing folder: {BASE_DIR}")
if not os.path.isdir(CLEAN_DATA_DIR):
    raise FileNotFoundError(f"Missing folder: {CLEAN_DATA_DIR}")

print(f'Using project root: {PROJECT_ROOT}')
print(f'Using raw data folder: {BASE_DIR}')
print(f'Using clean data folder: {CLEAN_DATA_DIR}')

Using project root: c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel
Using raw data folder: c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\raw_data
Using clean data folder: c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data


## OpenGo insoles
- remove 9 header rows
- remove unnecessary data
- resample from 100 Hz to 60 Hz by linear interpolation

In [28]:
# Data cleaner
# - Remove unnecessary data
# - Reshape data at 1/60-second intervals

data_path = os.path.join(BASE_DIR, 'Insoles')
output_path = CLEAN_DATA_DIR

if not os.path.isdir(data_path):
    raise FileNotFoundError(f"Missing folder: {data_path}")

csv_files = [f for f in os.listdir(data_path) if f.endswith('.txt')]

def resample_and_interpolate(data, interp_limit=6, edge_fill_limit=2):
    # Explicitly specify the timestamp column name
    timestamp_col = '# time'
    if timestamp_col not in data.columns:
        print(f"Error: Timestamp column '{timestamp_col}' does not exist in the data.")
        return None, None

    data = data.copy()
    data[timestamp_col] = pd.to_numeric(data[timestamp_col], errors='coerce')
    data = data.dropna(subset=[timestamp_col]).sort_values(timestamp_col)

    if data.empty:
        print("Error: No valid timestamp values were found.")
        return None, None

    # Force sensor columns to numeric before resampling
    sensor_cols = [c for c in data.columns if c != timestamp_col]
    data[sensor_cols] = data[sensor_cols].apply(pd.to_numeric, errors='coerce')

    na_before = int(data[sensor_cols].isna().sum().sum())

    # Use the time column as the resampling index in seconds
    data[timestamp_col] = pd.to_timedelta(data[timestamp_col], unit='s')
    data = data.set_index(timestamp_col)
    data = data[~data.index.duplicated(keep='first')]

    # Resample to 60 Hz
    data_resampled = data.resample('16.666667ms').mean()

    # Step 1: fill only short internal gaps
    data_resampled = data_resampled.interpolate(
        method='linear',
        limit=interp_limit,
        limit_direction='both',
        limit_area='inside'
    )
    na_after_interp = int(data_resampled.isna().sum().sum())

    # Step 2: cautiously fill short boundary gaps only
    if na_after_interp > 0:
        data_resampled = data_resampled.ffill(limit=edge_fill_limit).bfill(limit=edge_fill_limit)
    na_after_edge_fill = int(data_resampled.isna().sum().sum())

    # Step 3: last resort for any remaining sparse NaNs
    if na_after_edge_fill > 0:
        col_medians = data_resampled.median(numeric_only=True)
        data_resampled = data_resampled.fillna(col_medians)
    na_final = int(data_resampled.isna().sum().sum())

    # Convert the resampled index back to seconds
    data_resampled = data_resampled.reset_index()
    data_resampled[timestamp_col] = data_resampled[timestamp_col].dt.total_seconds()

    stats = {
        'na_before': na_before,
        'na_after_interp': na_after_interp,
        'na_after_edge_fill': na_after_edge_fill,
        'na_final': na_final,
    }

    return data_resampled, stats

for csv_file in csv_files:
    # Read CSV (9 rows in the header, tab-separated values)
    file_path = os.path.join(data_path, csv_file)
    data = pd.read_csv(file_path, header=9, sep='\t')

    # Normalize header formatting (some files include leading/trailing spaces)
    data.columns = data.columns.str.strip()

    # Remove unnecessary data (drop what exists, even if some columns are absent)
    cols_to_drop = [
        "left total force[N]", "left center of pressure X[-0.5...+0.5]", "left center of pressure Y[-0.5...+0.5]",
        "right total force[N]", "right center of pressure X[-0.5...+0.5]", "right center of pressure Y[-0.5...+0.5]",
        "right steps[]", "left steps[]"
    ]
    existing_cols = [c for c in cols_to_drop if c in data.columns]
    if existing_cols:
        data = data.drop(columns=existing_cols)
    else:
        print(f"Warning: No target columns found to drop in {csv_file}.")

    # Apply resampling + bounded NA handling
    data_resample, na_stats = resample_and_interpolate(data.copy())

    if data_resample is not None:
        # Save output file
        output_file_path = os.path.join(output_path, csv_file)
        data_resample.to_csv(output_file_path, index=False)

        # Confirm processing result
        print(f"Processed {csv_file}: removed {len(existing_cols)} columns")
        print(
            f"NA stats -> before: {na_stats['na_before']}, "
            f"after interp: {na_stats['na_after_interp']}, "
            f"after edge fill: {na_stats['na_after_edge_fill']}, "
            f"final: {na_stats['na_final']}"
        )
        print(data_resample.head())
    else:
        print(f'Error processing {csv_file}. Skipping.')


Processed Soles_001CcSs_2_tech_paos.txt: removed 6 columns
NA stats -> before: 3388, after interp: 22, after edge fill: 0, final: 0
     # time  left pressure 1[N/cm²]  left pressure 2[N/cm²]  \
0  0.000000                   5.875                   2.625   
1  0.016667                   5.625                   2.250   
2  0.033333                   5.875                   2.250   
3  0.050000                   6.750                   2.250   
4  0.066667                   6.500                   2.500   

   left pressure 3[N/cm²]  left pressure 4[N/cm²]  left pressure 5[N/cm²]  \
0                   3.500                   2.375                   6.750   
1                   3.375                   2.250                   6.875   
2                   3.375                   2.125                   6.750   
3                   3.500                   2.000                   6.750   
4                   3.500                   2.125                   6.250   

   left pressure 6[N/cm²] 

## Awinda IMMU
- export the joints positions tab of xlsx file into csv
- rename cols by X.#, Y.#, Z.#, ... with # a digit representing a joint

In [22]:
# Data cleaner
# - Read Segment Position tab directly from xlsx files
# - Change joint columns to X.0, Y.0, Z.0, X.1, ... , X.22, Y.22, Z.22

# Put raw data into the Awinda folder
data_path = os.path.join(BASE_DIR, 'Awinda')
output_path = CLEAN_DATA_DIR

if not os.path.isdir(data_path):
    raise FileNotFoundError(f"Missing folder: {data_path}")

# Get all xlsx files in the folder
xlsx_files = [f for f in os.listdir(data_path) if f.endswith('.xlsx')]


def rename_awinda_columns(data):
    coord_columns = [
        col for col in data.columns
        if re.search(r'\s[xyz]$', str(col), flags=re.IGNORECASE)
    ]
    expected_coord_columns = 23 * 3

    if len(coord_columns) != expected_coord_columns:
        raise ValueError(
            f'Expected {expected_coord_columns} joint coordinate columns, found {len(coord_columns)}.'
        )

    renamed_coord_columns = []
    for joint_idx in range(23):
        renamed_coord_columns.extend([f'X.{joint_idx}', f'Y.{joint_idx}', f'Z.{joint_idx}'])

    rename_map = dict(zip(coord_columns, renamed_coord_columns))
    return data.rename(columns=rename_map)


# Process each xlsx directly without creating temporary csv files
for xlsx_file in xlsx_files:
    xlsx_path = os.path.join(data_path, xlsx_file)

    try:
        xls = pd.ExcelFile(xlsx_path)
        if 'Segment Position' not in xls.sheet_names:
            print(f"Segment Position tab not found in {xlsx_file}. Skipping.")
            continue

        data = pd.read_excel(xls, sheet_name='Segment Position')
        data = rename_awinda_columns(data)
    except Exception as e:
        print(f'Error processing {xlsx_file}: {e}')
        continue

    output_file_name = os.path.splitext(xlsx_file)[0] + '.csv'
    output_file_path = os.path.join(output_path, output_file_name)
    data.to_csv(output_file_path, index=False, sep=',', decimal='.')

    print(f'Renamed columns for {output_file_name}:')
    print(data.head())


Exported Awinda_001CcSs_1_tech_vide.csv successfully.
Exported Awinda_001CcSs_2_tech_paos.csv successfully.
Exported Awinda_001CcSs_3_shadow.csv successfully.
Exported Awinda_001CcSs_4_leçon.csv successfully.
Exported Awinda_001CcSs_5_sparring.csv successfully.
Exported Awinda_002DdBb_1_tech_vide.csv successfully.
Exported Awinda_002DdBb_2_tech_paos.csv successfully.
Exported Awinda_002DdBb_3_shadow.csv successfully.
Exported Awinda_002DdBb_4_leçon.csv successfully.
Exported Awinda_002DdBb_5_sparring.csv successfully.
Exported Awinda_003MmRr_1_tech_vide.csv successfully.
Exported Awinda_003MmRr_2_tech_paos.csv successfully.
Exported Awinda_003MmRr_3_shadow.csv successfully.
Exported Awinda_003MmRr_4_leçon.csv successfully.
Exported Awinda_003MmRr_5_sparring.csv successfully.
Exported Awinda_004BbKk_1_tech_vide.csv successfully.
Exported Awinda_004BbKk_2_tech_paos.csv successfully.
Exported Awinda_004BbKk_3_shadow.csv successfully.
Exported Awinda_004BbKk_4_leçon.csv successfully.
Expor

In [23]:
# Position hold processing
# Use pelvis joint (X.0, Y.0, Z.0) to normalize/align skeleton coordinate columns

def normalize_skeleton_data(df):
    pelvis_cols = ['X.0', 'Y.0', 'Z.0']
    if not all(col in df.columns for col in pelvis_cols):
        raise ValueError(f'Missing pelvis columns: {pelvis_cols}')

    pelvis_data = df[pelvis_cols]
    normalized_df = df.copy()

    for col in df.columns:
        if col.startswith(('X.', 'Y.', 'Z.')):
            axis = col.split('.')[0]
            normalized_df[col] = df[col] - pelvis_data[f'{axis}.0']

    return normalized_df


# Get all csv files in the output folder
csv_files = [f for f in os.listdir(output_path) if f.endswith('.csv')]

# Process each CSV file in the clean_data directory and overwrite in place
for csv_file in csv_files:
    file_path = os.path.join(output_path, csv_file)
    try:
        data = pd.read_csv(file_path)
        normalized_data = normalize_skeleton_data(data)
    except Exception as e:
        print(f'Error normalizing {csv_file}: {e}')
        continue

    # Safe overwrite: write to temp file, then atomically replace original
    tmp_file_path = file_path + '.tmp'
    normalized_data.to_csv(tmp_file_path, index=False, sep=',', decimal='.')
    os.replace(tmp_file_path, file_path)

    print(f'Overwrote normalized skeleton data in {file_path}.')


Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_001CcSs_1_tech_vide.csv.
Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_001CcSs_2_tech_paos.csv.
Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_001CcSs_3_shadow.csv.
Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_001CcSs_4_leçon.csv.
Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_001CcSs_5_sparring.csv.
Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_002DdBb_1_tech_vide.csv.
Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projec